# Zero-shot CLIP ViT-B/32

Перед запуском:
```bash
.venv/bin/python -m src.evaluate --part train_sub
.venv/bin/python -m src.evaluate --part eval
```
Первая команда сравнивает варианты на train_sub и записывает выбор в `results/zs_choice.json`,
вторая считает только выбранный вариант на eval (val + test).

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, data, embed, evaluate, prompts, queries

plt.rcParams.update({"font.size": 12, "figure.dpi": 100})
M = evaluate.METRIC_COLS

## 1. Выбор варианта на train_sub

In [ ]:
variants = pd.read_csv(config.RESULTS / "zs_train_sub_variants.csv", index_col=[0, 1, 2])
variants[M + ["rand_ap"]].round(4)

Парный бутстрэп по запросам (10 000 повторов): разница mAP выбранного варианта со вторым и с вариантами,
где изменён один фактор, а остальные как у выбранного. Если 95% интервал накрывает ноль, на train_sub варианты не различаются,
и выбор первого - просто правило.

In [ ]:
boot = pd.read_csv(config.RESULTS / "zs_train_sub_bootstrap.csv")
boot.round(4)

In [ ]:
pq_train = pd.read_csv(config.RESULTS / "zs_train_sub_per_query.csv")
# каждый фактор при остальных равных: среднее AP по запросам, разбивка по типу запроса
for factor in ["mode", "template", "scoring"]:
    display(pq_train.pivot_table(index=factor, columns="n_attrs", values="ap", aggfunc="mean").round(4))

In [ ]:
choice = json.loads(evaluate.CHOICE_PATH.read_text())
choice

## 2. Метрики на eval

In [ ]:
pq = pd.read_csv(config.RESULTS / "zs_eval_per_query.csv")
by_type = pd.read_csv(config.RESULTS / "zs_eval_by_type.csv", index_col=0)
overall = evaluate.summary(pq)
overall.index = ["все"]
pd.concat([overall, by_type]).round(4)

In [ ]:
by_value = pd.read_csv(config.RESULTS / "zs_eval_by_value.csv", index_col=[0, 1])
by_value.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
labels = [f"{a}: {v}" for a, v in by_value.index]
y = np.arange(len(by_value))[::-1]
ax.barh(y, by_value.ap, color="#4c72b0", label="CLIP")
ax.scatter(by_value.rand_ap, y, color="black", marker="|", s=200, label="случайная выдача")
ax.set_yticks(y, labels)
ax.set_xlabel("mAP по запросам с этим значением, eval")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(config.RESULTS / "zs_eval_by_value.png", dpi=150)

## 3. Какой атрибут CLIP путает

Для запросов из 2 и 3 атрибутов: доля кадров в топ-50, у которых этот атрибут не совпал с запросом.

In [ ]:
miss = pq[pq.n_attrs > 1].groupby("n_attrs")[["miss_weather", "miss_scene", "miss_timeofday"]].mean()
miss.round(3)

## 4. Удачные и провальные запросы

Сравнивать запросы по сырому AP нельзя: у частых запросов он высокий и у случайной выдачи.
Поэтому сортирую по доле пути от случайной выдачи до идеальной: (AP - AP_rand) / (1 - AP_rand).

In [ ]:
pq["ap_norm"] = (pq.ap - pq.rand_ap) / (1 - pq.rand_ap)
cols = ["query_id", "n_relevant", "p@10", "p@50", "ap", "rand_ap", "ap_norm"]
ranked = pq.sort_values("ap_norm", ascending=False)[cols]
display(ranked.head(8).round(3))
display(ranked.tail(8).round(3))

Топ-6 кадров выдачи для двух лучших и двух худших запросов. Рамка зелёная, если кадр релевантен.
Картинки сохраняю в `artifacts/`, в репозиторий они не попадают.

In [ ]:
frames = data.load_frames("eval")
qs = queries.load_queries()
img = embed.load_image_embeddings(choice["mode"], frames)
model, processor = embed.load_model()
text_emb = evaluate.text_embeddings(model, processor, evaluate.all_texts(qs, choice["template"]))

show = list(ranked.query_id.head(2)) + list(ranked.query_id.tail(2))
sub_q = qs.set_index("query_id").loc[show].reset_index()
scores = evaluate.score_matrix(img, sub_q, choice["template"], choice["scoring"], text_emb)
N_TOP = 6
fig, axes = plt.subplots(len(show), N_TOP, figsize=(N_TOP * 3.2, len(show) * 2.3))
with zipfile.ZipFile(config.IMAGES_ZIP) as z:
    for i, q in enumerate(sub_q.to_dict("records")):
        g = np.flatnonzero(queries.gallery_mask(frames, q, config.GALLERY))
        top = g[np.argsort(-scores[g, i], kind="stable")[:N_TOP]]
        rel = queries.relevant_mask(frames, q)
        for j, idx in enumerate(top):
            r = frames.iloc[idx]
            ax = axes[i, j]
            ax.imshow(data.read_image(z, r.split, r["name"]).resize((320, 180)))
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_color("#2ca02c" if rel[idx] else "#d62728"); s.set_linewidth(4)
            ax.set_title(" / ".join(r[a] for a in config.ATTRS), fontsize=8)
        axes[i, 0].set_ylabel(prompts.query_text(q, "tags"), fontsize=10)
fig.tight_layout()
fig.savefig(config.ARTIFACTS / "zs_examples.png", dpi=110)